# Final Project Task 1

## Environment Requirements

This notebook uses:

- **Python version: 3.9**
- PyRadiomics
- Pandas
- NumPy

Note:
PyRadiomics is most stable with Python 3.7–3.10.  Python 3.9 is recommended to avoid installation issues.

---

## Objective

Task 1 : Extract Pyradiomics features from MRI data (already segmented) of 64 brain cancer patients (REMBRANDT) and generate a multi-sample feature matrix.


**Create Python 3.9 environment:**

*Note: Uses homebrew, but could be done with conda.*

In terminal:
* `brew install python@3.9` -- Install Python 3.9
* `python3.9 -m venv radiomics_env` -- Create virtual environment
* `source radiomics_env/bin/activate` -- Activate
* `python -m ipykernel install --user --name radiomics_env --display-name "Python 3.9 (radiomics)"` -- Add kernel to jupyter
* `jupyter notebook` -- Start jupyter notebook: 

In Jupyter notebook, change kernel to Python 3.9 (radiomics)

In [2]:
## Enforce Python 3.9

import sys

print("Python version:", sys.version)

if not sys.version.startswith("3.9"):
    raise EnvironmentError("This notebook requires Python 3.9. Please switch your environment.")

Python version: 3.9.25 (main, Oct 31 2025, 18:40:52) 
[Clang 17.0.0 (clang-1700.6.3.2)]


In [3]:
## Imports

# !pip install pyradiomics pandas
import os
import pandas as pd
from radiomics import featureextractor

In [4]:
## Define root directory

# Get curret working directory
# print(os.getcwd())

# Set the path to final project folder
rootDir = ".."

root_path = os.path.abspath(rootDir)
#print(root_path)

# Set the path to the data directory
dataDir = "../HIDS_7009_Final_project_data_sharing_master_copy"
data_path = os.path.abspath(dataDir)
#print(data_path)

# Set the paths to the Rembrandt data - patients divided into separate directories 
rembDir = "../HIDS_7009_Final_project_data_sharing_master_copy/Rembrandt_data/Rembrandt_Final_Data"
remb_path = os.path.abspath(rembDir)
print(remb_path)

# Print contents
print(os.listdir(remb_path))

/Users/stephaniearaki/Library/CloudStorage/GoogleDrive-sa2070@georgetown.edu/My Drive/HIDS/Fall25-Spring26/Spring-2026/HIDS-7009-ImagingInformatics/Assignments/final_project/HIDS_7009_Final_project_data_sharing_master_copy/Rembrandt_data/Rembrandt_Final_Data
['Patients_41_50', 'Patients_51_60', 'Patients_11_20', 'Patients_21_30', 'Patients_61_65', 'Patients_1_10', 'Patients_31_40']


In [5]:
## Initialize PyRadiomics feature extractor

# extracts quantitative features from image + label pairs.
extractor = featureextractor.RadiomicsFeatureExtractor()

# view enabled features
print("Enabled features:")
print(extractor.enabledFeatures)

Enabled features:
{'firstorder': [], 'glcm': [], 'gldm': [], 'glrlm': [], 'glszm': [], 'ngtdm': [], 'shape': []}


## Loop Through Patient Folders

We use `os.walk()` to:
- Traverse directories
- Identify image and label files
- Extract features for each patient

In [8]:
all_features = []
count = 0

# get total patients for progress tracking
## for this dataset, should only have 64
total_patients = sum(
    len(os.listdir(os.path.join(remb_path, s)))
    for s in os.listdir(remb_path)
    if os.path.isdir(os.path.join(remb_path, s))
)

for subset in os.listdir(remb_path):
    subset_path = os.path.join(remb_path, subset)
    
    if not os.path.isdir(subset_path):
        continue
    
    for patient in os.listdir(subset_path):
        patient_path = os.path.join(subset_path, patient)
        
        if not os.path.isdir(patient_path):
            continue
        
        imagePath = None
        labelPath = None
        
        for dirName, subdirList, fileList in os.walk(patient_path):
            
            for fname in fileList:
                fname_lower = fname.lower()

                # use only t1 image and avoid using t1ce
                if "t1" in fname_lower and "t1ce" not in fname_lower and fname_lower.endswith((".nii", ".nii.gz")):
                    imagePath = os.path.join(dirName, fname)
                elif "out-labels" in fname_lower and fname_lower.endswith((".nii", ".nii.gz")):
                    labelPath = os.path.join(dirName, fname)
        
        if imagePath and labelPath:
            count += 1
            print(f"[{count}/{total_patients}] Processing: {patient}")
            
            try:
                result = extractor.execute(imagePath, labelPath)
                features = dict(result)
                
                features["PatientID"] = os.path.basename(patient_path)
                
                all_features.append(features)
            
            except Exception as e:
                print(f"Error processing {patient_path}: {e}")

In [17]:
## Convert results to dataframe
df = pd.DataFrame(all_features)

# Remove diagnostic columns (cleaning)
df = df[[col for col in df.columns if not col.startswith("diagnostics")]]

# Preview
df.head()

In [18]:
# Check if PatientID is there
print(df.columns) 

In [12]:
# Move PatientID to front
cols = ["PatientID"] + [c for c in df.columns if c != "PatientID"]
df = df[cols]
print(df.columns)  # check if PatientID is there

In [16]:
# Expecting 64 rows
df.shape

In [13]:
## Save features to csv
output_path = "stephanie_araki_finalproject_step1_radiomics_features.csv"
df.to_csv(output_path, index=False)

print(f"Saved to {output_path}")